In [1]:
import os
import sys
import glob
import rasterio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../ProcessEvents/')
from config import CATCHMENT_LOOKUP_DICT, OUT_DIR # MOLLY_DIR_FF, RAINFALL_CSV_DIR, ENSEMBLE_MEMBERS, , CATCHMENTS

In [ ]:
### not sure why this one is missing

In [3]:
all_catchments = set(CATCHMENT_LOOKUP_DICT.keys())

catchments_with_flood_output = []
for catchment_num in all_catchments:
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/Catchment_{catchment_num}/{catchment_name}.pkl"
    flood_fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/5km_total/Catchment_{catchment_num}/Ens01_{catchment_num}/10cm/flooded_area_5km_total_Ens01_{catchment_num}_10cm.nc"
        
    if os.path.isfile(flood_fp) and os.path.isfile(fp):
        catchments_with_flood_output.append(catchment_num)
    else:
        print(catchment_num)
        print(os.path.isfile(fp))
        print(os.path.isfile(flood_fp))

89
False
True


In [4]:
# catchment_num = '25'
# catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
# if catchment_num in ['33_b', '26']:
#     fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/Catchment_{catchment_num}/{catchment_name}_new.pkl"
# else:
#     fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/Catchment_{catchment_num}/{catchment_name}.pkl"
# rainfall_events = pd.read_pickle(fp)
# fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/Catchment_{catchment_num}/all_events_added_soilvars.csv"
# rainfall_events = pd.read_csv(fp)
# rainfall_events.columns

In [6]:
rainfall_events_all = []
for catchment_num in catchments_with_flood_output:
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    if catchment_num in ['33_b', '26']:
        fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/Catchment_{catchment_num}/{catchment_name}_new.pkl"
    else:
        fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/Catchment_{catchment_num}/{catchment_name}.pkl"
    rainfall_events = pd.read_pickle(fp)
    if 'threshold_ante_rain_1d_mean_t80' in rainfall_events.columns:
        print("YES", catchment_num)
        rainfall_events.drop(['threshold_ante_rain_1d_mean_t80', 'cluster_ante_rain_1d_mean_t80',
       'threshold_ante_rain_2d_mean_t80', 'cluster_ante_rain_2d_mean_t80',
       'threshold_ante_rain_5d_mean_t80', 'cluster_ante_rain_5d_mean_t80',
       'threshold_ante_rain_10d_mean_t80', 'cluster_ante_rain_10d_mean_t80',
       'threshold_ante_sm_mean_t80', 'cluster_ante_sm_mean_t80',
                                   'threshold_ante_rain_1d_mean_t60', 'cluster_ante_rain_1d_mean_t60',
       'threshold_ante_rain_2d_mean_t60', 'cluster_ante_rain_2d_mean_t60',
       'threshold_ante_rain_5d_mean_t60', 'cluster_ante_rain_5d_mean_t60',
       'threshold_ante_rain_10d_mean_t60', 'cluster_ante_rain_10d_mean_t60',
                                   'threshold_ante_rain_2d_mean_t50', 'cluster_ante_rain_2d_mean_t50',
       'threshold_ante_rain_5d_mean_t50', 'cluster_ante_rain_5d_mean_t50',
       'threshold_ante_rain_10d_mean_t50', 'cluster_ante_rain_10d_mean_t50',
                            'neighbourhood_ante_rain_2d_mean', 'neighbourhood_ante_rain_2d_point',
       'neighbourhood_ante_rain_5d_mean', 'neighbourhood_ante_rain_5d_point',
       'neighbourhood_ante_rain_10d_mean', 'neighbourhood_ante_rain_10d_point',
                     'threshold_ante_sm_mean_t60', 'cluster_ante_sm_mean_t60',
                            'threshold_ante_rain_1d_mean_t50', 'cluster_ante_rain_1d_mean_t50',
       'threshold_ante_sm_mean_t50', 'cluster_ante_sm_mean_t50',      'neighbourhood_ante_rain_1d_mean', 'neighbourhood_ante_rain_1d_point',
       'neighbourhood_ante_sm_mean', 'neighbourhood_ante_sm_point',], axis=1, inplace=True)
    if 'start_month' in rainfall_events.columns:
        print("YES", catchment_num)
        del rainfall_events['start_day']
        del rainfall_events['start_hour']
        rainfall_events.rename(columns={'start_month': 'month'}, inplace=True)
        rainfall_events.rename(columns={'start_year':'start_year'}, inplace=True)
#         rainfall_events.rename(columns={'start_day':'day'}, inplace=True)
    #rainfall_events_complete = rainfall_events[rainfall_events['mismatch']!=True].copy()
    rainfall_events['catchment_num'] = catchment_num
    # print(f"Catchment {catchment_num} has {len(rainfall_events)}, of which {len(rainfall_events) - len(rainfall_events_complete)} are mismatched")
    rainfall_events_all.append(rainfall_events) 
rainfall_events_all_df = pd.concat(rainfall_events_all, ignore_index=True)   
rainfall_events_all_df = rainfall_events_all_df[rainfall_events_all_df['max_precip']<130].copy()

YES 40
YES 105
YES 26
YES 26
YES 33_b
YES 33_b
YES 23
YES 23


In [7]:
len(np.unique(rainfall_events_all_df['catchment_num']))

113

In [ ]:
### I think saving to csv cuts off some of the data

In [82]:
rainfall_events_all_df.to_pickle("/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/all_catchments.pkl")

In [ ]:
######## WITH ADDED SOIL VARS

In [38]:
rainfall_events_all = []
for catchment_num in catchments_with_flood_output:
    if catchment_num not in ['89', '46']:
        catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
        fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/Catchment_{catchment_num}/all_events_added_soilvars.csv"
        rainfall_events = pd.read_csv(fp)
        if 'lu_at_peak.2' in rainfall_events.columns:
            print("YES", catchment_num)
    #     if 'start_month' in rainfall_events.columns:
    #         print("YES", catchment_num)
    #         del rainfall_events['start_day']
    #         del rainfall_events['start_hour']
    #         rainfall_events.rename(columns={'start_month': 'month'}, inplace=True)
    #         rainfall_events.rename(columns={'start_year':'start_year'}, inplace=True)
    #         rainfall_events.rename(columns={'start_day':'day'}, inplace=True)
        #rainfall_events_complete = rainfall_events[rainfall_events['mismatch']!=True].copy()
        rainfall_events['catchment_num'] = catchment_num
        if len(rainfall_events) ==0:
            print(catchment_num)
        # print(f"Catchment {catchment_num} has {len(rainfall_events)}, of which {len(rainfall_events) - len(rainfall_events_complete)} are mismatched")
        rainfall_events_all.append(rainfall_events) 
rainfall_events_all_df = pd.concat(rainfall_events_all, ignore_index=True)   
# rainfall_events_all_df = rainfall_events_all_df[rainfall_events_all_df['max_precip']<130].copy()

91
71
79


/tmp/ipykernel_2589558/2810037324.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  rainfall_events_all_df = pd.concat(rainfall_events_all, ignore_index=True)


In [37]:
catchment_num = 46
fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/Catchment_{catchment_num}/all_events_added_soilvars.csv"
rainfall_events = pd.read_csv(fp)
rainfall_events

,max_precip,t_global,t_local,x_idx,y_idx,x_idx_global,y_idx_global,x_coord,y_coord,ens,...,se_max,se_min,fu_mean_m,fu_min_m,fu_max_m,sat_at_peak,fu_at_peak,fumax_at_peak,lu_at_peak,hc_at_peak.1
0,32.496731,5413,6,2,4,93,18,267500.0,57500.0,1,...,1.024863,0.0,0.007079,0.000000,0.016133,0.586082,0.007219,0.017116,0.038478,0.378609
1,40.201675,7556,16,0,8,91,22,257500.0,77500.0,1,...,1.254339,0.0,0.004033,0.000000,0.016133,0.980064,0.000333,0.015138,0.034985,0.314047
2,40.591404,7383,9,2,7,93,21,267500.0,72500.0,1,...,1.026706,0.0,0.006656,0.000000,0.016133,0.743806,0.004128,0.016104,0.037109,0.339249
3,66.453850,4572,3,1,6,92,20,262500.0,67500.0,1,...,1.207917,0.0,0.004478,0.000000,0.016133,0.730401,0.004408,0.016298,0.037433,0.345888
4,35.003792,6042,5,1,9,92,23,262500.0,82500.0,1,...,0.969685,0.0,0.007618,0.000189,0.016341,0.466060,0.008741,0.016377,0.037800,0.356855
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1393,33.033485,7801,14,5,4,96,18,282500.0,57500.0,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1394,30.487280,4259,2,1,10,92,24,262500.0,87500.0,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1395,36.359234,7004,9,2,9,93,23,267500.0,82500.0,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1396,71.383133,6132,3,1,2,92,16,262500.0,47500.0,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
catchment_num='59'
catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/Catchment_{catchment_num}/all_events_added_soilvars.csv"
rainfall_events = pd.read_csv(fp)
if 'lu_at_peak.2' in rainfall_events.columns:
    print("YES", catchment_num)
#     if 'start_month' in rainfall_events.columns:
#         print("YES", catchment_num)
#         del rainfall_events['start_day']
#         del rainfall_events['start_hour']
#         rainfall_events.rename(columns={'start_month': 'month'}, inplace=True)
#         rainfall_events.rename(columns={'start_year':'start_year'}, inplace=True)
#         rainfall_events.rename(columns={'start_day':'day'}, inplace=True)
#rainfall_events_complete = rainfall_events[rainfall_events['mismatch']!=True].copy()
rainfall_events['catchment_num'] = catchment_num
rainfall_events

,max_precip,t_global,t_local,x_idx,y_idx,x_idx_global,y_idx_global,x_coord,y_coord,ens,...,cluster_flood_30_area_t80,threshold_flood_30_vol_t80,cluster_flood_30_vol_t80,catchment_num,peak_cell_slope_avg,peak_cell_slope_max,catchment_slope_avg,catchment_slope_max,peak_cell_sink_frac,length


In [58]:
# dupes = rainfall_events_all_df[rainfall_events_all_df.duplicated(subset=["x_coord", "y_coord"],
#         keep=False)].sort_values(["x_coord", "y_coord"])

# dupes[['month', 'lu_at_peak']][:50]

In [11]:
nulls = rainfall_events_all_df[rainfall_events_all_df['fu_at_peak'].isnull()][['catchment_num', 'ens', 'lu_at_peak', 'event_num']]
nulls

,catchment_num,ens,lu_at_peak,event_num
88958,46,1,NaN,56
88962,46,1,NaN,60
88976,46,1,NaN,74
89031,46,1,NaN,129
89032,46,1,NaN,130
...,...,...,...,...
90296,46,15,NaN,141
90297,46,15,NaN,142
90298,46,15,NaN,143
90299,46,15,NaN,144


In [39]:
len(rainfall_events_all_df.loc[~rainfall_events_all_df["catchment_num"].map(lambda x: isinstance(x, (int, float))), "catchment_num"].unique())

109

In [107]:
rainfall_events_all_df.to_pickle("/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/all_catchments_added_soilvars.pkl")